# Setup

In [1]:
!pip install mujoco gymnasium matplotlib imageio numpy torch torchvision pandas open_clip_torch tqdm transformers timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 11.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
os.chdir("/content/drive/MyDrive/visual-policy-learning")
os.environ["MUJOCO_GL"] = "egl"

# Rendering


In [4]:
import importlib
import h5py
import numpy as np
from tqdm import tqdm

import envs.panda_reach_env

importlib.reload(envs.panda_reach_env)

from envs.panda_reach_env import PandaReachEnv


In [ ]:
input_path = "data/panda_demos.h5"
output_path = "data/panda_demos_rendered.h5"

env = PandaReachEnv(
    render_mode=True,
    image_width=64,
    image_height=64,
    physics_steps=4,
)

stride = 3

with h5py.File(input_path, "r") as f:
    states = f["states"][::stride]
    actions = f["actions"][::stride]
    success = f["success"][::stride]
    rewards = f["rewards"][::stride]
    dones = f["dones"][::stride]
    has_targets = "targets" in f
    targets = f["targets"][::stride] if has_targets else None

images = np.zeros((len(states), 64, 64, 3), dtype=np.uint8)

print("Rendering...")

for i, state in enumerate(tqdm(states)):
    env.set_state(state)
    if targets is not None:
        env.set_target(targets[i])
    images[i] = env.render()

print("Saving...")

with h5py.File(output_path, "w") as f:
    f.create_dataset("observations", data=images)
    f.create_dataset("actions", data=actions)
    f.create_dataset("states", data=states)
    if targets is not None:
        f.create_dataset("targets", data=targets)
    f.create_dataset("success", data=success)
    f.create_dataset("rewards", data=rewards)
    f.create_dataset("dones", data=dones)

env.close()
print("Done")


Rendering...


100%|██████████| 1541/1541 [09:14<00:00,  2.78it/s]


Saving...
Done


# Feature Extraction

In [5]:
import torch
import torch.nn.functional as F
import torchvision.models as models
import open_clip
from transformers import AutoModel, AutoImageProcessor
import h5py
import numpy as np
from tqdm import tqdm
import timm

In [ ]:
input_path = "data/panda_demos_rendered.h5"
output_path = "data/panda_resnet_features.h5"

device = "cuda" if torch.cuda.is_available() else "cpu"

weights = models.ResNet18_Weights.IMAGENET1K_V1

resnet = models.resnet18(weights=weights)

# Remove classifier head
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])

resnet = resnet.to(device)
resnet.eval()

# ImageNet normalization
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

with h5py.File(input_path, "r") as f:
    images = f["observations"][:]
    actions = f["actions"][:]

batch_size = 64
features = []

print("Extracting ResNet18 features...")

for i in tqdm(range(0, len(images), batch_size)):
    batch = images[i:i + batch_size]

    batch = (
        torch.from_numpy(batch)
        .permute(0, 3, 1, 2)
        .float()
        / 255.0
    )

    batch = F.interpolate(
        batch,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
    )

    batch = (batch - mean) / std
    batch = batch.to(device)

    with torch.inference_mode():
        feat = resnet(batch)
        feat = feat.squeeze(-1).squeeze(-1)

    features.append(feat.cpu().numpy())

features = np.concatenate(features, axis=0)

with h5py.File(output_path, "w") as f:
    f.create_dataset("features", data=features)
    f.create_dataset("actions", data=actions)

print("Saved ResNet18 features ✔")
print("Feature shape:", features.shape)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 411MB/s]


Extracting ResNet18 features...


100%|██████████| 25/25 [01:18<00:00,  3.15s/it]


Saved ResNet18 features ✔
Feature shape: (1541, 512)


In [ ]:
input_path = "data/panda_demos_rendered.h5"
output_path = "data/panda_clip_features.h5"

device = "cuda" if torch.cuda.is_available() else "cpu"

model, _, _ = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

model = model.to(device)
model.eval()

# CLIP normalization
mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1)
std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1)

with h5py.File(input_path, "r") as f:
    images = f["observations"][:]
    actions = f["actions"][:]

batch_size = 64
features = []

print("Extracting CLIP features...")

for i in tqdm(range(0, len(images), batch_size)):
    batch = images[i:i + batch_size]

    batch = (
        torch.from_numpy(batch)
        .permute(0, 3, 1, 2)
        .float()
        / 255.0
    )

    batch = F.interpolate(
        batch,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
    )

    batch = (batch - mean) / std
    batch = batch.to(device)

    with torch.inference_mode():
        feat = model.encode_image(batch)

        # Recommended for CLIP embeddings
        feat = feat / feat.norm(dim=-1, keepdim=True)

    features.append(feat.cpu().numpy())

features = np.concatenate(features, axis=0)

with h5py.File(output_path, "w") as f:
    f.create_dataset("features", data=features)
    f.create_dataset("actions", data=actions)

print("Saved CLIP features ✔")
print("Feature shape:", features.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Extracting CLIP features...


100%|██████████| 25/25 [02:58<00:00,  7.13s/it]


Saved CLIP features ✔
Feature shape: (1541, 512)


In [6]:
input_path = "data/panda_demos_rendered.h5"
output_path = "data/panda_dino_features.h5"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model(
    "vit_small_patch14_dinov2.lvd142m",
    pretrained=True,
    num_classes=0,
    img_size=224,
)

model = model.to(device)
model.eval()

# ImageNet normalization
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

with h5py.File(input_path, "r") as f:
    images = f["observations"][:]
    actions = f["actions"][:]

batch_size = 32
features = []

print("Extracting DINOv2 features...")

for i in tqdm(range(0, len(images), batch_size)):
    batch = images[i:i + batch_size]

    batch = (
        torch.from_numpy(batch)
        .permute(0, 3, 1, 2)
        .float()
        / 255.0
    )

    batch = F.interpolate(
        batch,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
    )

    batch = (batch - mean) / std
    batch = batch.to(device)

    with torch.inference_mode():
        feat = model(batch)
        feat = feat / feat.norm(dim=-1, keepdim=True)

    features.append(feat.cpu().numpy())

features = np.concatenate(features, axis=0)

with h5py.File(output_path, "w") as f:
    f.create_dataset("features", data=features)
    f.create_dataset("actions", data=actions)

print("Saved DINOv2 features ✔")
print("Feature shape:", features.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Extracting DINOv2 features...


100%|██████████| 49/49 [06:42<00:00,  8.21s/it]


Saved DINOv2 features ✔
Feature shape: (1541, 384)


# Frozen Training

In [ ]:
!python training/train_bc_frozen.py \
  --data data/panda_resnet_features.h5 \
  --model_name resnet_frozen \
  --action_dim 8 \
  --epochs 200 \
  --save_every 10 \
  --batch_size 64 \
  --lr 1e-4


Training: resnet_frozen
Input dim: 512
Train samples: 1232 | Val samples: 309

[000] train=0.1950 val=0.1208 best=0.1208 lr=1.00e-04
[001] train=0.1047 val=0.0757 best=0.0757 lr=1.00e-04
[002] train=0.0791 val=0.0606 best=0.0606 lr=9.99e-05
[003] train=0.0687 val=0.0526 best=0.0526 lr=9.99e-05
[004] train=0.0619 val=0.0512 best=0.0512 lr=9.98e-05
[005] train=0.0572 val=0.0473 best=0.0473 lr=9.98e-05
[006] train=0.0526 val=0.0433 best=0.0433 lr=9.97e-05
[007] train=0.0487 val=0.0415 best=0.0415 lr=9.96e-05
[008] train=0.0465 val=0.0385 best=0.0385 lr=9.95e-05
[009] train=0.0439 val=0.0379 best=0.0379 lr=9.94e-05
[010] train=0.0416 val=0.0367 best=0.0367 lr=9.93e-05
[011] train=0.0405 val=0.0347 best=0.0347 lr=9.91e-05
[012] train=0.0376 val=0.0341 best=0.0341 lr=9.90e-05
[013] train=0.0368 val=0.0333 best=0.0333 lr=9.88e-05
[014] train=0.0364 val=0.0332 best=0.0332 lr=9.86e-05
[015] train=0.0354 val=0.0319 best=0.0319 lr=9.84e-05
[016] train=0.0345 val=0.0312 best=0.0312 lr=9.82e-05
[0

In [ ]:
!python training/train_bc_frozen.py \
  --data data/panda_clip_features.h5 \
  --model_name clip_frozen \
  --action_dim 8 \
  --epochs 200 \
  --save_every 10 \
  --batch_size 64 \
  --lr 1e-4


Training: clip_frozen
Input dim: 512
Train samples: 1232 | Val samples: 309

[000] train=0.1855 val=0.1254 best=0.1254 lr=1.00e-04
[001] train=0.1192 val=0.0964 best=0.0964 lr=1.00e-04
[002] train=0.0975 val=0.0812 best=0.0812 lr=9.99e-05
[003] train=0.0854 val=0.0715 best=0.0715 lr=9.99e-05
[004] train=0.0800 val=0.0679 best=0.0679 lr=9.98e-05
[005] train=0.0738 val=0.0630 best=0.0630 lr=9.98e-05
[006] train=0.0712 val=0.0600 best=0.0600 lr=9.97e-05
[007] train=0.0678 val=0.0574 best=0.0574 lr=9.96e-05
[008] train=0.0658 val=0.0552 best=0.0552 lr=9.95e-05
[009] train=0.0626 val=0.0537 best=0.0537 lr=9.94e-05
[010] train=0.0611 val=0.0507 best=0.0507 lr=9.93e-05
[011] train=0.0586 val=0.0492 best=0.0492 lr=9.91e-05
[012] train=0.0562 val=0.0483 best=0.0483 lr=9.90e-05
[013] train=0.0543 val=0.0471 best=0.0471 lr=9.88e-05
[014] train=0.0543 val=0.0446 best=0.0446 lr=9.86e-05
[015] train=0.0511 val=0.0468 best=0.0446 lr=9.84e-05
[016] train=0.0498 val=0.0441 best=0.0441 lr=9.82e-05
[017

In [7]:
!python training/train_bc_frozen.py \
  --data data/panda_dino_features.h5 \
  --model_name dino_frozen \
  --action_dim 8 \
  --epochs 200 \
  --save_every 10 \
  --batch_size 64 \
  --lr 1e-4


Training: dino_frozen
Input dim: 384
Train samples: 1232 | Val samples: 309

[000] train=0.1789 val=0.1131 best=0.1131 lr=1.00e-04
[001] train=0.1019 val=0.0738 best=0.0738 lr=1.00e-04
[002] train=0.0784 val=0.0645 best=0.0645 lr=9.99e-05
[003] train=0.0711 val=0.0573 best=0.0573 lr=9.99e-05
[004] train=0.0636 val=0.0526 best=0.0526 lr=9.98e-05
[005] train=0.0607 val=0.0489 best=0.0489 lr=9.98e-05
[006] train=0.0568 val=0.0459 best=0.0459 lr=9.97e-05
[007] train=0.0530 val=0.0453 best=0.0453 lr=9.96e-05
[008] train=0.0503 val=0.0432 best=0.0432 lr=9.95e-05
[009] train=0.0494 val=0.0417 best=0.0417 lr=9.94e-05
[010] train=0.0482 val=0.0416 best=0.0416 lr=9.93e-05
[011] train=0.0458 val=0.0393 best=0.0393 lr=9.91e-05
[012] train=0.0440 val=0.0398 best=0.0393 lr=9.90e-05
[013] train=0.0439 val=0.0369 best=0.0369 lr=9.88e-05
[014] train=0.0418 val=0.0364 best=0.0364 lr=9.86e-05
[015] train=0.0405 val=0.0380 best=0.0364 lr=9.84e-05
[016] train=0.0410 val=0.0345 best=0.0345 lr=9.82e-05
[017

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def plot_run(path, label):
    df = pd.read_csv(path)

    epochs = df["epoch"]
    train = df["train_loss"]
    val = df["val_loss"]

    plt.plot(epochs, train, linestyle="--", alpha=0.7, label=f"{label} Train")
    plt.plot(epochs, val, linewidth=2, label=f"{label} Val")


plt.figure(figsize=(10, 6))

plot_run("models/checkpoints/resnet_frozen/logs.csv", "ResNet18")
plot_run("models/checkpoints/clip_frozen/logs.csv", "CLIP")
plot_run("models/checkpoints/dino_frozen/logs.csv", "DINO")

plt.legend()
plt.title("Behavior Cloning: Train vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Smooth L1 Loss")
plt.grid(True, alpha=0.3)
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'models/checkpoints/resnet_frozen/logs.csv'

<Figure size 1000x600 with 0 Axes>